<a href="https://colab.research.google.com/github/gandersen-ctb/skilljar/blob/main/Accessing_Claude_with_the_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Making a Request

Install Dependencies.

In [2]:
%pip install anthropic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 6.2 MB/s eta 0:00:00


Load ENV variables...

In [3]:
from dotenv import load_dotenv

load_dotenv()

False

Import the rehusable methods and variables.

In [4]:
import sys
sys.path.append('/content/') # Add the directory where shared.py is located

from shared import model, max_tokens, chat, add_role_message
import shared # Import shared module directly to access its attributes like __file__

print(shared.__file__)

/content/shared.py


Create an API Client instance:

In [5]:
from anthropic import Anthropic
client = Anthropic(
    # This is the default and can be omitted
    # api_key=os.environ.get("ANTHROPIC_API_KEY"),
)

Make the request:

In [6]:
response = client.messages.create(
    model = model,
    max_tokens = max_tokens,
    messages = [
        # List of messages to send, structured with the role, user vs assistant, second for models, first for users.
        {"role": "user", "content": "What is the meaning of life ?"}, # Message from a normal user.
        # {"role": "assistant", "content": "Here is the computer vission result..."}, # Message from the AI assistant.
    ]
)

TypeError: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

In [ ]:
response2 = client.messages.create(
    model = model,
    max_tokens = max_tokens,
    messages = [
        # Proving API wont keep the context of the conversation unless explicitly built.
        {"role": "user", "content": "Do you agree with that ?"}, # Follow up message
    ]
)

Formated Response:

In [ ]:
print(response.content[0].text)

print("----------------------------------------------------------------------------")
print("-------------------------------- RESPONSE 2 --------------------------------")
print("----------------------------------------------------------------------------")

print(response2.content[0].text)

# Multi-Turn conversations

Make a starting list of messages

In [ ]:
messages = []

Add in the initial user question.

In [ ]:
add_role_message(messages, 'user', 'What is the meaning of life ?')

Pass the list of messages

In [ ]:
answer = chat(client, messages)
print(answer)

Take the answer and add it as assistant message in history.

In [ ]:
add_role_message(messages, 'assistant', answer)

# Print the messages
messages

Add follow-up question to the message list.

In [ ]:
add_role_message(messages, 'user', 'Do you agree with that ?')

Call API again with contextual messages and see response difference.

In [ ]:
answer = chat(client, messages)
print(answer)

# Chat Exercise

Make the initial list of messages

In [ ]:
messages=[]

Use a while true endless loop to keep prompting the user for input

# System prompts



Math problem no system promts

In [ ]:
messages = []
add_role_message(messages, "user", "How do I solve 5x+3=2 for x?")

answer_no_system = chat(client, messages)

system ="""
You are a patient math tutor.
Do not directly answer the student's questions.
Guide them to a solution step by step.
"""
answer_system = chat(client, messages, system)
print("----------------------------------------------------------------------------")
print("------------------------------ WITHOUT SYTEM -------------------------------")
print(answer_no_system)

print("----------------------------------------------------------------------------")
print("-------------------------------- WITH SYTEM --------------------------------")
print(answer_system)

Clean code return examples

In [ ]:
while True:
  # Get the input
  user_input = input(">")
  print("User >", user_input)

  # Append
  add_role_message(messages, 'user', user_input)
  # Call Claude
  answer = chat(client, messages)
  # Append Response to the history
  add_role_message(messages, 'assistant', answer)
  # Print the answer
  print("Claude >", answer)



In [ ]:
messages =[]
add_role_message(messages, "user", "Write a python function that checks a string for duplicate characters.")
answer_no_system = chat(client, messages)

# Adding system prompt.
system = """ You are tasked with writting concise explicit code """
answer_system = chat(client, messages, system)


print("----------------------------------------------------------------------------")
print("------------------------------ WITHOUT SYTEM -------------------------------")
print(answer_no_system)

print("----------------------------------------------------------------------------")
print("-------------------------------- WITH SYTEM --------------------------------")
print(answer_system)

# Temperature

In [ ]:
messages = []
add_role_message(messages, 'user', "Generate a one sentence movie idea")
answer_low_temperature = chat(client, messages, temperature=0.0)

print("----------------------------------------------------------------------------")
print("------------------------------ LOW TEMPERATURE -----------------------------")
print(answer_low_temperature)

answer_high_temperature = chat(client, messages, temperature=1.0)

print("----------------------------------------------------------------------------")
print("------------------------------ HIGH TEMPERATURE ----------------------------")
print(answer_high_temperature)

# Response streaming

In [ ]:
# Manually support the streams in the API.
messages = []
add_role_message(messages, 'user', "Write a 1 sentence description of a fake database")
stream = client.messages.create(
    model=model,
    max_tokens=max_tokens,
    messages=messages,
    stream=True
)

for event in stream:
  print(event)

In [ ]:
# Natively support for streams in the API.
messages = []
add_role_message(messages, 'user', "Write a 1 sentence description of a fake database")
with client.messages.stream(
    model=model,
    max_tokens=max_tokens,
    messages=messages
) as stream:
  for text in stream.text_stream:
    print(text, end="")
    pass

stream.get_final_message() # get the final message once the stream ends

# Structured data

In [ ]:
# Manually support the streams in the API.
messages = []
add_role_message(messages, 'user', "Generate a very short event bridge rule as json")
add_role_message(messages, 'assistant', "```json")
chat(client, messages, stop_sequences=["```"])
